In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# ================================
#  Baseline: ElasticNet + Purged Time CV + EWMA sizing
#  - Dict-based configs (no dataclass)
#  - Auto-detects train/test under /kaggle/input
#  - Produces /kaggle/working/allocations.csv
# ================================
from pathlib import Path
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from inspect import signature


import os, sys, json, time, threading, logging, traceback
from typing import List, Dict, Any, Optional


/kaggle/input/hull-tactical-market-prediction/train.csv
/kaggle/input/hull-tactical-market-prediction/test.csv
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_inference_server.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/templates.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/base_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/relay.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/kaggle_evaluation.proto
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2_grpc.py
/kaggl

In [2]:
# -----------------------------
# CONFIG — EDIT ME (minimally)
# -----------------------------
ID_COL      = "row_id"         # <- the evaluator's id field name
TARGET_COL  = "y"              # <- name you used during training; not in inference JSON
DATE_COL    = None             # e.g. "date" if you used it in build_features
GROUP_COL   = None             # e.g. "ticker" if you used panel features

# Where your trained artifacts live (upload as a Kaggle Dataset and add as input)
# Provide at least one! The code will work if only one is present.
LGB_PATH    = "/kaggle/input/hull-models/lgb_model.txt"   # LightGBM Booster.save_model path
XGB_PATH    = "/kaggle/input/hull-models/xgb_model.json"  # XGBoost Booster.save_model path
FEAT_PATH   = "/kaggle/input/hull-models/feature_list.json"  # Optional: list[str] feature order
BLEND_W     = None  # e.g., 0.42 for w*LGB + (1-w)*XGB. If None, will try analytic weight if available.

# -----------------------------
# Logging (view in /working after rerun)
# -----------------------------
logging.basicConfig(
    filename="/kaggle/working/server.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)
log = logging.getLogger("hull_server")

# -----------------------------
# Try to import your helpers
# -----------------------------
def has_func(name: str) -> bool:
    return name in globals()

# Your notebook likely defines these:
# - basic_clean(train, test, id_col, target, date_col, group_col)
# - build_features(df, id_col, target, date_col, group_col)
# - analytic_weight(oof_a, oof_b, y)  (optional)
# - train_lgb / train_xgb (not needed at inference)

# -----------------------------
# Load artifacts (fast, robust)
# -----------------------------
ART: Dict[str, Any] = {}
LOAD_ERROR: Optional[Exception] = None

def _load_json_safe(path: str):
    try:
        if path and os.path.exists(path):
            with open(path, "r") as f:
                return json.load(f)
    except Exception:
        log.exception("Failed loading json: %s", path)
    return None

def load_artifacts() -> Dict[str, Any]:
    out: Dict[str, Any] = {}
    # Feature order (optional but recommended)
    feature_list = _load_json_safe(FEAT_PATH)

    # LightGBM
    lgb_booster = None
    try:
        if LGB_PATH and os.path.exists(LGB_PATH):
            import lightgbm as lgb
            lgb_booster = lgb.Booster(model_file=LGB_PATH)
            log.info("Loaded LGB from %s", LGB_PATH)
    except Exception:
        log.exception("Failed to load LGB model")

    # XGBoost
    xgb_booster = None
    try:
        if XGB_PATH and os.path.exists(XGB_PATH):
            import xgboost as xgb
            xgb_booster = xgb.Booster()
            xgb_booster.load_model(XGB_PATH)
            log.info("Loaded XGB from %s", XGB_PATH)
    except Exception:
        log.exception("Failed to load XGB model")

    out["feature_list"] = feature_list
    out["lgb"] = lgb_booster
    out["xgb"] = xgb_booster
    out["blend_w"] = BLEND_W
    return out

try:
    ART = load_artifacts()
    log.info("Artifacts loaded: keys=%s", list(ART.keys()))
except Exception as e:
    LOAD_ERROR = e
    log.exception("Artifact load failed")

In [3]:
# -----------------------------
# Feature building (reusing YOUR code when present)
# -----------------------------
def _coerce_df(instances: List[Dict[str, Any]]) -> pd.DataFrame:
    """Turn evaluator JSON rows into a DataFrame."""
    df = pd.DataFrame(instances)
    # Guarantee ID col exists
    if ID_COL not in df.columns:
        # Create a fallback incremental id if evaluator didn't send it (rare)
        df[ID_COL] = np.arange(len(df), dtype=np.int64)
    return df

def build_features_for_inference(instances: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Uses your build_features(df, id_col, target, date_col, group_col) if available,
    otherwise does a minimal pass-through and drops non-numeric columns.
    """
    df = _coerce_df(instances)

    # If your notebook has a "basic_clean" that expects train/test, fake it:
    if has_func("basic_clean"):
        try:
            # Create a dummy 'train' with 0 rows to let your cleaner infer schema,
            # and treat incoming as 'test'.
            train_dummy = pd.DataFrame(columns=[c for c in df.columns if c != TARGET_COL])
            train_clean, test_clean = basic_clean(
                train_dummy, df.copy(), ID_COL, TARGET_COL, DATE_COL, GROUP_COL
            )
            df = test_clean
        except Exception:
            log.exception("basic_clean failed; continuing without it")

    if has_func("build_features"):
        try:
            feats = build_features(
                df.copy(), ID_COL, TARGET_COL, DATE_COL, GROUP_COL
            )
        except Exception:
            log.exception("build_features failed; fallback to numeric-only")
            feats = df.select_dtypes(include=[np.number]).copy()
    else:
        # Fallback: numeric-only (ensures inference runs even if your function name differs)
        feats = df.select_dtypes(include=[np.number]).copy()

    # Remove target if it somehow exists
    if TARGET_COL in feats.columns:
        feats = feats.drop(columns=[TARGET_COL])

    # Enforce feature order if provided
    feature_list = ART.get("feature_list")
    if feature_list:
        for col in feature_list:
            if col not in feats.columns:
                feats[col] = 0.0
        feats = feats[feature_list]

    # Final NaN handling
    feats = feats.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return feats

In [4]:
# -----------------------------
# Prediction (LGB + XGB blend)
# -----------------------------
def _predict_lgb(X: pd.DataFrame):
    booster = ART.get("lgb")
    if booster is None:
        return None
    import lightgbm as lgb
    ds = lgb.Dataset(X, free_raw_data=False)
    # Booster.predict can be called directly on numpy, but Dataset ensures float32
    return booster.predict(X.values, num_iteration=getattr(booster, "best_iteration", None))

def _predict_xgb(X: pd.DataFrame):
    booster = ART.get("xgb")
    if booster is None:
        return None
    import xgboost as xgb
    dmat = xgb.DMatrix(X.values, feature_names=list(X.columns))
    return booster.predict(dmat)

def _blend(p_lgb, p_xgb, w: Optional[float]):
    if p_lgb is None and p_xgb is None:
        # no models loaded, return zeros to keep server alive (bad score but debuggable)
        return np.zeros(0, dtype="float32")
    if p_lgb is None:
        return p_xgb
    if p_xgb is None:
        return p_lgb
    if w is None:
        # Try to compute analytic weight if your function exists AND we have some proxy.
        # In production you’d compute this on OOF; during inference we must pick a fixed w.
        try:
            # If your notebook defines analytic_weight(oof_a, oof_b, y) we **cannot**
            # call it here (no ground-truth y). Fall back to a reasonable constant.
            w = 0.5
        except Exception:
            w = 0.5
    return w * p_lgb + (1.0 - w) * p_xgb

def predict_batch(instances: List[Dict[str, Any]]) -> Dict[str, Any]:
    df_raw = _coerce_df(instances)
    X = build_features_for_inference(instances)

    # Ensure float32 matrix
    X = X.astype("float32")

    p_lgb = _predict_lgb(X)
    p_xgb = _predict_xgb(X)

    preds = _blend(p_lgb, p_xgb, ART.get("blend_w"))
    if preds is None or len(preds) == 0:
        preds = np.zeros(len(df_raw), dtype="float32")

    out = [{"row_id": int(df_raw.iloc[i][ID_COL]), "y": float(preds[i])}
           for i in range(len(df_raw))]
    return {"predictions": out}

# -----------------------------
# Flask server (/ping, /predict)
# -----------------------------
from flask import Flask, request, jsonify
app = Flask(__name__)

@app.get("/ping")
def ping():
    ok = (LOAD_ERROR is None) and (ART is not None)
    return jsonify({"status": "ok" if ok else "error",
                    "detail": None if ok else str(LOAD_ERROR)}), (200 if ok else 500)

@app.post("/predict")
def predict_endpoint():
    try:
        if LOAD_ERROR is not None or ART is None:
            return jsonify({"error": f"Model not ready: {LOAD_ERROR}"}), 500
        payload = request.get_json(force=True, silent=False)
        instances = payload.get("instances", [])
        result = predict_batch(instances)
        return jsonify(result), 200
    except Exception as e:
        log.exception("Error in /predict")
        return jsonify({"error": str(e), "trace": traceback.format_exc()}), 500

def _run_server():
    port = int(os.environ.get("PORT", "8000"))
    app.run(host="0.0.0.0", port=port, debug=False, use_reloader=False, threaded=True)


In [5]:
# -----------------------------
# Start server in background
# -----------------------------
import requests
t = threading.Thread(target=_run_server, daemon=True)
t.start()

PORT = int(os.environ.get("PORT", "8000"))
for _ in range(60):
    try:
        r = requests.get(f"http://127.0.0.1:{PORT}/ping", timeout=1.5)
        if r.status_code == 200:
            print("✅ Server healthy")
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Inference server failed to start in time.")

 * Serving Flask app '__main__'
 * Debug mode: off
✅ Server healthy
